# 03b — TabPFN, on Colab

The tabular foundation model of the comparison. TabPFN needs a GPU to be practical, so it lives here
rather than in `03_models.ipynb`.

**How to run it**

1. run `03_models.ipynb` locally first — it writes `data/processed/model_table_folds.csv`;
2. open this notebook in Colab and set *Runtime → Change runtime type → T4 GPU*;
3. run the cells; the upload cell asks for `model_table_folds.csv`;
4. the last cell downloads `tabpfn_oof.csv`. Put it in `data/processed/` and re-run the last cell of
   `03_models.ipynb` to merge it in.

The fold numbers travel inside the file, so TabPFN is scored on exactly the same split as the
logistic regression and XGBoost. Nothing here recomputes a split.

In [ ]:
!pip install tabpfn -q

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, or expect about ten minutes a fold.")

## 1. The data

In Colab this opens a file picker: choose `model_table_folds.csv`. Outside Colab it reads the file
from the repository, so the notebook can also be run locally if the GPU is not available.

In [ ]:
import numpy as np
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()
    table = pd.read_csv(next(iter(uploaded)))
except ImportError:
    table = pd.read_csv("data/processed/model_table_folds.csv")

print(table.shape, "| folds:", sorted(table.fold.unique()))

## 2. Same preparation as the other two models

One-hot for the six category codes, median imputation for the rest, fitted on the training fold only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

CATEGORICAL = ["w_field_cd", "w_race", "w_goal", "m_field_cd", "m_race", "m_goal"]

X = table.drop(columns=["pair", "wave", "match", "fold"])
y = table.match

def preparation():
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), CATEGORICAL)],
        remainder="passthrough")
    return make_pipeline(encode, SimpleImputer(strategy="median"))

## 3. Fit and predict, fold by fold

TabPFN does not train in the usual sense: it carries the training rows and makes its predictions in a
single forward pass, so a fold takes seconds on a GPU. `n_estimators` is how many internal ensemble
members it averages.

Two settings worth knowing about, both tested:

- `ignore_pretraining_limits=True` is only needed off the GPU. TabPFN refuses to run on a CPU with
  more than 1,000 rows, and we have 3,337 per training fold. On a GPU the flag changes nothing; on a
  CPU it makes the notebook run, slowly — count roughly ten minutes a fold.
- if the GPU runs out of memory, lower `n_estimators` to 4 or 2 before anything else.

In [ ]:
import time
from tabpfn import TabPFNClassifier

predictions = pd.Series(np.nan, index=table.index)

for k in sorted(table.fold.unique()):
    train, test = table.fold != k, table.fold == k
    prep = preparation()

    started = time.time()
    model = TabPFNClassifier(n_estimators=8, random_state=0, ignore_pretraining_limits=True)
    model.fit(prep.fit_transform(X[train]), y[train])
    predictions[test] = model.predict_proba(prep.transform(X[test]))[:, 1]
    print(f"fold {k}: {train.sum()} train / {test.sum()} test, {time.time() - started:.0f}s")

print("missing predictions:", int(predictions.isna().sum()))

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

print("ROC-AUC:", round(roc_auc_score(y, predictions), 3))
print("PR-AUC: ", round(average_precision_score(y, predictions), 3))

## 4. Send it back

`tabpfn_oof.csv` holds one row per pair, keyed on `pair` so `03_models.ipynb` can join it to the
other two models' predictions.

In [ ]:
out = pd.DataFrame({"pair": table.pair, "tabpfn": predictions.to_numpy()})
out.to_csv("tabpfn_oof.csv", index=False)

try:
    from google.colab import files
    files.download("tabpfn_oof.csv")
except ImportError:
    print("saved tabpfn_oof.csv next to this notebook")

out.head(3)